In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import polars.selectors as cs

from src.data.utils.sparsity import build_ohlcv_sparsity_report
from src.stats.vif import variance_inflation_factor
from src.technical_indicators import add_indicators

In [2]:
base_path = Path("../data/coinbase/ohlcv")

In [3]:
btc_1m = pl.read_parquet(f"{base_path}/btc-usdc/1m/2025/01.parquet")
btc_5m = pl.read_parquet(f"{base_path}/btc-usdc/5m/2025/01.parquet")

In [ ]:

df = pl.read_parquet("../data/coinbase/ohlcv/aave-usdc/5m/2025/01.parquet")

# Multiple indicators at once
indicators = [
    ("adx", "5m", {"timeperiod": 14}),
    ("adx", "15m", {"timeperiod": 5}),
    ("rsi", "5m", {"timeperiod": 7}),
    ("rsi", "15m", {"timeperiod": 14}),
    ]
df_with_features = add_indicators(df, indicators, base_timeframe="5m")

In [5]:
df_with_features.tail()

timestamp,open,high,low,close,volume,adx_14_5m,adx_5_15m,rsi_7_5m,rsi_14_15m
"datetime[μs, UTC]",f64,f64,f64,f64,f64,f64,f64,f64,f64
2025-01-31 23:35:00 UTC,333.2,333.24,332.4,332.84,257.435,15.641569,36.89097,34.914945,41.50759
2025-01-31 23:40:00 UTC,332.81,333.26,332.6,332.6,17.862,15.857324,33.29547,33.545954,40.782354
2025-01-31 23:45:00 UTC,332.45,333.0,331.78,332.91,83.021,16.4592,34.822617,37.253416,42.187461
2025-01-31 23:50:00 UTC,333.02,333.38,331.74,332.35,339.089,16.7221,34.306821,33.334044,40.325998
2025-01-31 23:55:00 UTC,332.51,332.66,331.96,332.61,123.6,16.966222,33.894184,36.928363,41.614096


In [6]:
df_with_features = df_with_features.drop_nans()
df_num = df_with_features.select(cs.numeric())

In [7]:
df_num.head()

open,high,low,close,volume,adx_14_5m,adx_5_15m,rsi_7_5m,rsi_14_15m
f64,f64,f64,f64,f64,f64,f64,f64,f64
314.57,315.23,313.78,314.82,77.475,62.19861,87.605531,69.636701,68.125362
314.84,315.33,313.95,315.18,55.639,61.879506,89.700131,72.242091,69.46007
315.33,316.2,314.81,315.89,132.7,61.933311,91.451939,76.81888,71.954361
315.83,316.4,314.63,314.94,167.06,62.057761,92.898334,61.094074,64.37791
314.92,315.83,314.85,315.37,81.23,62.173321,94.055451,64.889385,66.116993


In [8]:
vif = variance_inflation_factor(df=df_num)

In [9]:
vif.head(100)

feature,VIF
str,f64
"""open""",2080.145441
"""high""",2379.913849
"""low""",2401.80434
"""close""",2419.0634
"""volume""",1.409688
"""adx_14_5m""",1.416934
"""adx_5_15m""",1.39976
"""rsi_7_5m""",9.425955
"""rsi_14_15m""",8.625385


In [ ]:
report = build_ohlcv_sparsity_report(
    base_path=base_path,
    # assets=["near-usdc"],
)

In [ ]:
filtered_report = report.filter(
    pl.col("availability") > 0.8
)
filtered_report.head(100)

In [ ]:
df = pl.read_parquet(f"{base_path}/near-usdc/1m/2026/01.parquet")

In [ ]:
# Pass the timestamp series as X and the close series as Y
df = df.sort("timestamp")
plt.plot(df["timestamp"], df["close"])

plt.xlabel("timestamp")
plt.ylabel("price")
plt.title("Plotting shit")
plt.show()

In [ ]:
import warnings
warnings.filterwarnings('always')

vif = variance_inflation_factor(df_with_features)
vif